# Meta-learning: few-shot learning of NEW classes (and do we forget the old?)

The right test for meta-learning: how fast can a model learn brand-NEW classes from only a few
clips? We start from the ResNet50 trained on the 10 base classes, then adapt it to 10 UNSEEN
classes (TASK_1 + TASK_2) from just K shots each - and also check how much it FORGETS the old 10.

- **No meta** - take the trained model and few-shot fine-tune it to the new classes.
- **MAML** - first meta-train it on N-way K-shot episodes of the OLD classes (learning to adapt
  fast), then few-shot adapt to the new classes.

Both start from the same model and use the same few shots; only MAML meta-trains first. We report
NEW-class accuracy (did it learn them) and OLD-class retention (did it forget), for each arm.

## **Connect Colab**

In [1]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [2]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [3]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

Drive mounted.
Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
ULAL_DATASET_ROOT = /kaggle/input/ucf101-action-recognition/
ULAL_OUTPUT_ROOT  = /kaggle/working/ucf101-processed/
ULAL_DATA_ROOT     = /content/UCF101
ULAL_DRIVE_PROJECT = /content/drive/MyDrive/apai
Setup complete — no config.py edit, no kernel restart.


In [4]:
!pip install -q yt-dlp

In [5]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

ULAL_ROOT : /content/ulal
Device    : cuda
Classes   : base=10 task1=5 task2=5


42

## **Setup**

In [ ]:
OLD_N      = len(cfg.SELECTED_CLASSES)   # 10 known classes the ResNet50 was trained on
N_WAY      = 5                           # classes per few-shot episode (each episode tests 5 of the new)
K_SHOT     = 5                           # shots per class - the small-sample limit; try 1 for a bigger gap
K_QUERY    = 5
EVAL_DRAWS = 5                           # meta-test episodes to average (each also checks old-class retention)

META_EPOCHS   = 5                        # episodic meta-training (MAML arm only)
META_EPISODES = 30
INNER_LR      = 0.01
INNER_STEPS   = 5
META_LR       = 1e-4                      # gentle: a large meta-lr degrades the strong ResNet50 init
print(f"{N_WAY}-way {K_SHOT}-shot | new classes = TASK_1 + TASK_2 (10 unseen)")

## **Preprocess data**

The 10 known classes (for the meta-training episodes + the old-class retention test) and the 10 new
classes (TASK_1 + TASK_2, unseen).

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [ ]:
# KNOWN 10  (max_samples caps TRAIN clips/class via cfg.MAX_SAMPLES; None = full data)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# NEW (TASK 1)
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# NEW (TASK 2)
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx = {c: i for i, c in enumerate(ALL_CLASSES_LIST)}
print(f"{OLD_N} known + {len(cfg.TASK_1) + len(cfg.TASK_2)} new classes")

## **Create Loaders**

An old-class test loader (for the retention check) plus RAM pools: the KNOWN classes for the
meta-training episodes, and the NEW classes for the meta-test episodes.

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
old_test_loader = DataLoader(base_test, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
pool_old_clips, pool_old_labels = build_pool([base_train], per_class=20)                # known 10 -> meta-train
pool_new_clips, pool_new_labels = build_pool([task1_train, task2_train], per_class=20)  # new 10  -> meta-test
old_ids = sorted(set(pool_old_labels.tolist()))
new_ids = sorted(set(pool_new_labels.tolist()))
print(f"old pool {tuple(pool_old_clips.shape)} classes {len(old_ids)} | new pool {tuple(pool_new_clips.shape)} classes {len(new_ids)}")

## **Load the ResNet50 (knows the 10 old classes)**

The stage-2 ResNet50 (~93% on the base 10). We keep its trained old-class head aside to measure
retention later. (Run backbone_transfer first so the checkpoint exists in Drive.)

In [ ]:
set_seed(cfg.SEED)
base_model = ResNet50LSTM(num_classes=OLD_N).to(device)
base_model.load_state_dict(torch.load(cfg.RESNET50_PATH, map_location=device))
old_head = copy.deepcopy(base_model.fc)     # trained 10-way head - used to measure old-class retention
print("loaded ResNet50 (knows the 10 old classes)")

## **Old-class accuracy BEFORE any adaptation (retention reference)**

In [ ]:
old_acc_before, _, _ = test_model(base_model, old_test_loader, device)
print(f"OLD-class test accuracy before adapting to anything: {old_acc_before:.4f}")

## **Common init**

Give the model a fresh N-way head for the new classes. Both arms copy this exact model, so they
start identical - the only later difference is MAML's meta-training.

In [ ]:
init_model = copy.deepcopy(base_model)
init_model.fc = nn.Linear(cfg.HIDDEN_SIZE, N_WAY).to(device)
print(init_model.fc)

## **No meta: few-shot fine-tune to the NEW classes**

Take the trained model and adapt it to K shots of the new classes the normal way. Each draw also
re-checks the old-class accuracy through the adapted backbone (retention).

In [ ]:
base_new_acc, base_old_ret = [], []
for d in range(EVAL_DRAWS):
    set_seed(cfg.SEED + d)     # draw d reproducible + identical in the MAML arm
    sx, sy, qx, qy = sample_episode(pool_new_clips, pool_new_labels, new_ids, N_WAY, K_SHOT, K_QUERY, device)
    na, orr = adapt_new_and_retain(copy.deepcopy(init_model), sx, sy, qx, qy,
                                   old_head, old_test_loader, INNER_LR, INNER_STEPS, device)
    base_new_acc.append(na); base_old_ret.append(orr)
    torch.cuda.empty_cache()
print(f"no-meta  NEW acc {np.mean(base_new_acc):.3f}  |  OLD retention {np.mean(base_old_ret):.3f}")

## **MAML: meta-train on the OLD classes, then adapt the NEW ones**

Meta-train the shared init on N-way K-shot episodes of the KNOWN classes so it learns to adapt
fast (gentle META_LR + frozen backbone keep the features intact). Then adapt the SAME new-class
episodes as the no-meta arm.

In [ ]:
set_seed(cfg.SEED)
maml_model = copy.deepcopy(init_model)
maml_model, hist_maml = meta_train_maml(maml_model, pool_old_clips, pool_old_labels, old_ids,
                                        N_WAY, K_SHOT, K_QUERY, META_EPOCHS, META_EPISODES,
                                        INNER_LR, INNER_STEPS, META_LR, device, freeze_backbone=True)

In [ ]:
maml_new_acc, maml_old_ret = [], []
for d in range(EVAL_DRAWS):
    set_seed(cfg.SEED + d)     # SAME seed per draw -> SAME new-class episodes as the no-meta arm
    sx, sy, qx, qy = sample_episode(pool_new_clips, pool_new_labels, new_ids, N_WAY, K_SHOT, K_QUERY, device)
    na, orr = adapt_new_and_retain(copy.deepcopy(maml_model), sx, sy, qx, qy,
                                   old_head, old_test_loader, INNER_LR, INNER_STEPS, device)
    maml_new_acc.append(na); maml_old_ret.append(orr)
    torch.cuda.empty_cache()
print(f"MAML     NEW acc {np.mean(maml_new_acc):.3f}  |  OLD retention {np.mean(maml_old_ret):.3f}")
print(f"MAML gain on NEW classes: {np.mean(maml_new_acc) - np.mean(base_new_acc):+.3f}")

## **Training history (MAML meta-training)**

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, META_EPOCHS + 1), hist_maml["query_accs"], marker="s")
plt.xlabel("meta epoch"); plt.ylabel("meta query accuracy"); plt.ylim(0, 1.05)
plt.title("MAML meta-training progress (episodes from the OLD classes)")
plt.tight_layout(); plt.show()

## **Compare: learning NEW classes vs forgetting OLD**

Left: how well each arm learns the new classes from K shots (higher = better, MAML should win).
Right: old-class accuracy - 'before' vs each arm after adapting (a drop = forgetting).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.bar(["no meta", "MAML"], [np.mean(base_new_acc), np.mean(maml_new_acc)],
        yerr=[np.std(base_new_acc), np.std(maml_new_acc)], capsize=8)
ax1.set_ylim(0, 1.05); ax1.set_ylabel("new-class few-shot accuracy")
ax1.set_title(f"Learning NEW classes ({N_WAY}-way {K_SHOT}-shot)")
ax2.bar(["before", "no meta", "MAML"], [old_acc_before, np.mean(base_old_ret), np.mean(maml_old_ret)])
ax2.set_ylim(0, 1.05); ax2.set_ylabel("old-class accuracy")
ax2.set_title("OLD-class retention (drop from 'before' = forgetting)")
plt.tight_layout(); plt.show()

In [ ]:
print(f"{'method':<10}{'NEW acc':>10}{'OLD retention':>15}{'forgetting':>12}")
print(f"{'no meta':<10}{np.mean(base_new_acc):>10.3f}{np.mean(base_old_ret):>15.3f}{old_acc_before - np.mean(base_old_ret):>12.3f}")
print(f"{'MAML':<10}{np.mean(maml_new_acc):>10.3f}{np.mean(maml_old_ret):>15.3f}{old_acc_before - np.mean(maml_old_ret):>12.3f}")

## **Save**

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"n_way": N_WAY, "k_shot": K_SHOT, "eval_draws": EVAL_DRAWS,
           "old_acc_before": old_acc_before,
           "no_meta_new": float(np.mean(base_new_acc)), "no_meta_old_ret": float(np.mean(base_old_ret)),
           "maml_new": float(np.mean(maml_new_acc)),   "maml_old_ret": float(np.mean(maml_old_ret)),
           "maml_curve": hist_maml["query_accs"]}
with open(f"{cfg.RESULTS_DIR}/stage5_meta_fewshot.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/stage5_meta_fewshot.json")